# 15 — Silver — Identidade Empresarial

## Purpose
Build `silver_identidade` — the canonical supplier identity table joining
Receita Federal establishment, company, and Simples Nacional data with all
domain reference tables.

Filtered by ADR-007: only CNPJs present in procurement or sanctions sources
are included (~190k root CNPJs out of 70M total — 0.26% of the RF universe).

## Input
| Source | Path | Rows |
|---|---|---|
| `rf_estabelecimentos` | `data/bronze/rf_estabelecimentos/` | 70,145,812 |
| `rf_empresas` | `data/bronze/rf_empresas/` | 67,642,315 |
| `rf_simples` | `data/bronze/rf_simples/` | 48,097,045 |
| Domain tables | `data/bronze/rf_*/` | < 10k each |
| `silver_ceis` | `data/silver/silver_ceis/data.parquet` | ~27k |
| `silver_cnep` | `data/silver/silver_cnep/data.parquet` | ~3k |
| `silver_compras` | `data/silver/silver_compras/**/*.parquet` | ~711k |
| `silver_pncp` | `data/silver/silver_pncp/data.parquet` | ~3.9M |

## Output
- `data/silver/silver_identidade/` — Parquet partitioned by `uf`
- Grain: 1 row per CNPJ-14 (establishment level)
- Scope: only CNPJs whose `cnpj_basico` appears in any Silver source (ADR-007)

## Steps
1. Imports and configuration
2. Materialize Bronze RF tables into bronze.duckdb (skip if already loaded)
3. Build cnpjs_ativos from Silver sources (ADR-007 filter)
4. Build silver_identidade — loop by UF
5. Validation

## Notes
- Anchor: `rf_estabelecimentos` — all JOINs are LEFT to preserve scope
- ADR-002: CNPJ always VARCHAR — never cast to INTEGER
- ADR-005: no pseudonymization here — cnpj_token applied at Silver→Gold boundary
- ADR-007: filter at Silver layer — Bronze remains complete and unfiltered
- Sentinels: BIGINT 0 → NULL (dates); `'00000000'` → NULL (rf_simples dates); `'8888888'` → NULL (CNAE)
- PII excluded: correio_eletronico, telefone_1/2, ddd_1/2, ddd_fax, fax
- Idempotent by UF: skips partitions already written — delete dir to force full rerun

## Step 1 - Imports and configuration

In [ ]:
# =============================================================================
# Step 1 — Imports and configuration
# =============================================================================

import sys
import shutil
from pathlib import Path
from datetime import datetime, timezone

# --- utils -------------------------------------------------------------------
try:
    _nb_file = Path(__vsc_ipynb_file__).resolve()
    _root_candidate = _nb_file.parent.parent
except NameError:
    _root_candidate = Path.cwd().parent

sys.path.insert(0, str(_root_candidate / "utils"))

from paths        import get_project_root, bronze_path, silver_path, ensure_dir, to_sql_path
from duckdb_utils import get_connection, scalar, table_exists, register_parquet_view, safe_date_expr
from validation   import CheckSuite

import duckdb

# --- Paths -------------------------------------------------------------------
ROOT    = get_project_root()
DB_FILE = ROOT / "data" / "bronze.duckdb"
OUT_DIR = ROOT / "data" / "silver" / "silver_identidade"

BD = to_sql_path(ROOT / "data" / "bronze")
SD = to_sql_path(OUT_DIR)

# Silver sources — para construção do filtro ADR-007
SRC_CEIS    = silver_path(ROOT, "silver_ceis")
SRC_CNEP    = silver_path(ROOT, "silver_cnep")
SRC_COMPRAS = to_sql_path(ROOT / "data" / "silver" / "silver_compras") + "/**/*.parquet"
SRC_PNCP    = silver_path(ROOT, "silver_pncp")

STARTED_AT = datetime.now(timezone.utc)

# Idempotent — always rebuild
# Reason: cnpjs_ativos is rebuilt from Silver sources on every run.
# If upstream Silver changes (new CEIS records, new contracts), partition-level
# skip would leave stale UF files inconsistent with the new cnpjs_ativos scope.
if OUT_DIR.exists():
    shutil.rmtree(OUT_DIR)
    print(f"Deleted existing output: {OUT_DIR}")
ensure_dir(OUT_DIR)

print(f"ROOT     : {ROOT}")
print(f"DB_FILE  : {DB_FILE}")
print(f"OUT_DIR  : {OUT_DIR}")
print(f"Started  : {STARTED_AT.isoformat()}")
print(f"DuckDB   : {duckdb.__version__}")
print()
print("Configuration ready.")

## Step 2 - Load domain tables

Domain tables are small (< 10k rows each). We register them as DuckDB views
so the main query can reference them by name without repeating file paths.

In [ ]:
# =============================================================================
# Step 2 — Materialize Bronze RF tables into bronze.duckdb
#
# Strategy: persistent .duckdb file avoids rebuilding the hash table on every
# UF iteration. Skip if tables already exist with data — safe for reruns.
# Delete bronze.duckdb manually to force a full reload.
# =============================================================================

print("=" * 60)
print("  STEP 2 — Materializing Bronze RF into bronze.duckdb")
print("=" * 60)
print()

con = get_connection(
    db_path=DB_FILE,
    memory_limit="4GB",
    preserve_insertion_order=False,
)

sources = [
    ("rf_estabelecimentos", f"{BD}/rf_estabelecimentos/**/*.parquet"),
    ("rf_empresas",         f"{BD}/rf_empresas/**/*.parquet"),
    ("rf_simples",          f"{BD}/rf_simples/**/*.parquet"),
]

for table_name, parquet_path in sources:
    if table_exists(con, table_name):
        n = scalar(con, f"SELECT count(*) FROM {table_name}")
        if n > 0:
            print(f"  {table_name:<30} {n:>15,} rows  SKIPPED (already loaded)")
            continue

    t0 = datetime.now()
    print(f"  Loading {table_name}...")
    con.execute(f"DROP TABLE IF EXISTS {table_name}")
    con.execute(f"""
        CREATE TABLE {table_name} AS
        SELECT * FROM read_parquet('{parquet_path}')
    """)
    n       = scalar(con, f"SELECT count(*) FROM {table_name}")
    elapsed = (datetime.now() - t0).total_seconds()
    size_str = f"{DB_FILE.stat().st_size / (1024**3):.1f}GB" if DB_FILE.exists() else "criando..."
    print(f"  {table_name:<30} {n:>15,} rows  {elapsed:>6.1f}s  db={size_str}")

print()
if DB_FILE.exists():
    print(f"  bronze.duckdb size: {DB_FILE.stat().st_size / (1024**3):.1f} GB")
else:
    print(f"  bronze.duckdb: sendo criado...")

# Domain tables — registered as views (small, no need to materialize)
domains = {
    "dom_naturezas":     f"{BD}/rf_naturezas/**/*.parquet",
    "dom_qualificacoes": f"{BD}/rf_qualificacoes/**/*.parquet",
    "dom_cnaes":         f"{BD}/rf_cnaes/**/*.parquet",
    "dom_municipios":    f"{BD}/rf_municipios/**/*.parquet",
    "dom_paises":        f"{BD}/rf_paises/**/*.parquet",
    "dom_motivos":       f"{BD}/rf_motivos/**/*.parquet",
}

print()
for name, path in domains.items():
    register_parquet_view(con, name, path)
    n = scalar(con, f"SELECT count(*) FROM {name}")
    print(f"  {name:<25} {n:>6,} rows  OK")

print()
print("  Step 2 complete.")

## Step 3 - Build silver_identidade

Main transformation query following the Silver contract (EDA notebook, section 10).
Key decisions:
- Anchor: `rf_estabelecimentos` - all 70.1M rows preserved
- All JOINs are LEFT - no establishment lost due to missing domain code
- `cnpj_normalized`: `cnpj_basico || cnpj_ordem || cnpj_dv`
- BIGINT dates: `CASE WHEN val = 0 THEN NULL ELSE CAST(CAST(val AS VARCHAR) AS DATE) END`
- VARCHAR dates (rf_simples): `CASE WHEN val = '00000000' THEN NULL ELSE CAST(val AS DATE) END`
- `capital_social`: `REPLACE(',', '.') -> CAST AS DECIMAL(15,2)`
- `porte_empresa` and `situacao_cadastral`: inline MAP - no JOIN needed
- CNAE sentinel '8888888': mapped to NULL

In [ ]:
# =============================================================================
# Step 3 — Build cnpjs_ativos (ADR-007 filter)
#
# Reads cnpj_basico from all four Silver sources already produced by
# notebooks 13 and 14. Materializes a deduplicated list of active root CNPJs
# into the bronze.duckdb connection for use in Step 4.
#
# Using cnpj_basico (8 digits) instead of cnpj_normalized (14 digits) ensures
# all establishments of an active company are included, even if the specific
# branch in the contract differs from the one in Receita Federal.
# =============================================================================

print("=" * 60)
print("  STEP 3 — Building cnpjs_ativos (ADR-007)")
print("=" * 60)
print()

t0 = datetime.now()

# Drop and rebuild — always fresh to reflect latest Silver state
con.execute("DROP TABLE IF EXISTS cnpjs_ativos")
con.execute(f"""
    CREATE TABLE cnpjs_ativos AS
    SELECT DISTINCT cnpj_basico
    FROM (
        SELECT cnpj_basico FROM read_parquet('{SRC_CEIS}')
        UNION ALL
        SELECT cnpj_basico FROM read_parquet('{SRC_CNEP}')
        UNION ALL
        SELECT cnpj_basico FROM read_parquet('{SRC_COMPRAS}')
        UNION ALL
        SELECT cnpj_basico FROM read_parquet('{SRC_PNCP}')
    )
    WHERE cnpj_basico IS NOT NULL
""")

n_ativos = scalar(con, "SELECT count(*) FROM cnpjs_ativos")
elapsed  = (datetime.now() - t0).total_seconds()

print(f"  cnpjs_ativos built in {elapsed:.1f}s")
print(f"  Distinct cnpj_basico (root CNPJs) : {n_ativos:,}")
print()

# Sanity check — how many RF establishments match?
n_match = scalar(con, """
    SELECT count(*)
    FROM rf_estabelecimentos e
    WHERE e.cnpj_basico IN (SELECT cnpj_basico FROM cnpjs_ativos)
""")
pct = round(n_match / scalar(con, "SELECT count(*) FROM rf_estabelecimentos") * 100, 2)
print(f"  RF establishments in scope : {n_match:,}  ({pct}% of 70M total)")
print()
print("  Step 3 complete.")

In [ ]:
# =============================================================================
# Step 4 — Build silver_identidade — loop by UF
#
# Reads from bronze.duckdb native tables (fast hash joins — no Parquet reopen).
# Applies ADR-007 filter via cnpjs_ativos in cte_est.
# Writes one Parquet file per UF — idempotent (skips existing partitions).
# =============================================================================

print("=" * 60)
print("  STEP 4 — silver_identidade transformation by UF")
print("=" * 60)
print()

# Date expression helpers (safe_date_expr from duckdb_utils)
# BIGINT dates in rf_estabelecimentos: cast to VARCHAR first, sentinel = '0'
_dt_sit_cad = """
        CASE WHEN data_situacao_cadastral = 0 THEN NULL
             ELSE TRY_STRPTIME(CAST(data_situacao_cadastral AS VARCHAR), '%Y%m%d')::DATE
        END"""

_dt_inicio  = """
        CASE WHEN data_inicio_atividade = 0 THEN NULL
             ELSE TRY_STRPTIME(CAST(data_inicio_atividade AS VARCHAR), '%Y%m%d')::DATE
        END"""

_dt_sit_esp = """
        CASE WHEN data_situacao_especial = 0 THEN NULL
             ELSE TRY_STRPTIME(CAST(data_situacao_especial AS VARCHAR), '%Y%m%d')::DATE
        END"""

# VARCHAR dates in rf_simples: sentinel = '00000000'
_dt_opc_sim = """
        CASE WHEN data_opcao_simples = 0 THEN NULL
             ELSE TRY_STRPTIME(CAST(data_opcao_simples AS VARCHAR), '%Y%m%d')::DATE
        END"""
_dt_exc_sim = safe_date_expr("data_exclusao_simples", "'00000000'", "'%Y%m%d'")
_dt_opc_mei = safe_date_expr("data_opcao_mei",        "'00000000'", "'%Y%m%d'")
_dt_exc_mei = safe_date_expr("data_exclusao_mei",     "'00000000'", "'%Y%m%d'")

def build_sql_uf(uf: str, out_file: str) -> str:
    return f"""
    COPY (
        WITH
        cte_est AS (
            SELECT
                cnpj_basico || cnpj_ordem || cnpj_dv                   AS cnpj_normalized,
                cnpj_basico, cnpj_ordem, cnpj_dv,
                CASE identificador_matriz_filial
                    WHEN 1 THEN 'Matriz' WHEN 2 THEN 'Filial'
                    ELSE 'Desconhecido'
                END                                                     AS tipo_estabelecimento,
                nome_fantasia,
                situacao_cadastral                                      AS situacao_cadastral_cod,
                CASE situacao_cadastral
                    WHEN '01' THEN 'Nula'
                    WHEN '02' THEN 'Ativa'
                    WHEN '03' THEN 'Suspensa'
                    WHEN '04' THEN 'Inapta'
                    WHEN '05' THEN 'Ativa Nao Regular'
                    WHEN '08' THEN 'Baixada'
                    WHEN '99' THEN 'Exclusao Logica'
                    ELSE 'Desconhecida'
                END                                                     AS situacao_cadastral_desc,
                {_dt_sit_cad}                                           AS data_situacao_cadastral,
                motivo_situacao_cadastral                               AS motivo_situacao_cadastral_cod,
                nome_cidade_exterior,
                pais                                                    AS pais_cod,
                {_dt_inicio}                                            AS data_inicio_atividade,
                CASE WHEN cnae_fiscal_principal = '8888888' THEN NULL
                     ELSE cnae_fiscal_principal
                END                                                     AS cnae_fiscal_principal_cod,
                cnae_fiscal_secundaria,
                tipo_logradouro, logradouro, numero, complemento,
                bairro, cep,
                municipio                                               AS municipio_cod,
                situacao_especial,
                {_dt_sit_esp}                                           AS data_situacao_especial
            FROM rf_estabelecimentos
            WHERE uf = '{uf}'
              AND cnpj_basico IN (SELECT cnpj_basico FROM cnpjs_ativos)
        ),
        cte_emp AS (
            SELECT
                cnpj_basico, razao_social,
                natureza_juridica                                       AS natureza_juridica_cod,
                qualificacao_responsavel                                AS qualificacao_responsavel_cod,
                TRY_CAST(REPLACE(capital_social, ',', '.') AS DECIMAL(15,2)) AS capital_social,
                CASE porte_empresa
                    WHEN '01' THEN 'Micro Empresa'
                    WHEN '03' THEN 'Empresa de Pequeno Porte'
                    WHEN '05' THEN 'Demais'
                    ELSE NULL
                END                                                     AS porte_empresa,
                ente_federativo_responsavel
            FROM rf_empresas
        ),
        cte_sim AS (
            SELECT
                cnpj_basico,
                CASE opcao_simples WHEN 'S' THEN TRUE WHEN 'N' THEN FALSE ELSE NULL END AS opcao_simples,
                CASE opcao_mei     WHEN 'S' THEN TRUE WHEN 'N' THEN FALSE ELSE NULL END AS opcao_mei,
                {_dt_opc_sim}                                           AS data_opcao_simples,
                {_dt_exc_sim}                                           AS data_exclusao_simples,
                {_dt_opc_mei}                                           AS data_opcao_mei,
                {_dt_exc_mei}                                           AS data_exclusao_mei
            FROM rf_simples
        )
        SELECT
            est.cnpj_normalized,
            est.cnpj_basico, est.cnpj_ordem, est.cnpj_dv,
            est.tipo_estabelecimento,
            est.nome_fantasia,
            est.situacao_cadastral_cod, est.situacao_cadastral_desc,
            est.data_situacao_cadastral,
            est.motivo_situacao_cadastral_cod,
            mot.motivo_descricao                                        AS motivo_situacao_cadastral_desc,
            est.data_inicio_atividade,
            est.cnae_fiscal_principal_cod,
            cnae.cnae_descricao                                         AS cnae_fiscal_principal_desc,
            est.cnae_fiscal_secundaria,
            est.tipo_logradouro, est.logradouro, est.numero,
            est.complemento, est.bairro, est.cep,
            '{uf}'                                                      AS uf,
            est.municipio_cod,
            mun.municipio_descricao                                     AS municipio_desc,
            est.nome_cidade_exterior,
            est.pais_cod,
            pais.pais_descricao                                         AS pais_desc,
            est.situacao_especial, est.data_situacao_especial,
            emp.razao_social,
            emp.natureza_juridica_cod,
            nat.natureza_descricao                                      AS natureza_juridica_desc,
            emp.qualificacao_responsavel_cod,
            COALESCE(qual.qualificacao_descricao, 'Nao encontrada na tabela de dominio')
                                                                        AS qualificacao_responsavel_desc,
            emp.capital_social, emp.porte_empresa,
            emp.ente_federativo_responsavel,
            sim.opcao_simples, sim.data_opcao_simples, sim.data_exclusao_simples,
            sim.opcao_mei,     sim.data_opcao_mei,     sim.data_exclusao_mei,
            CURRENT_TIMESTAMP                                           AS _silver_loaded_at
        FROM cte_est est
        LEFT JOIN cte_emp          emp  ON est.cnpj_basico              = emp.cnpj_basico
        LEFT JOIN cte_sim          sim  ON est.cnpj_basico              = sim.cnpj_basico
        LEFT JOIN dom_naturezas    nat  ON emp.natureza_juridica_cod     = nat.natureza_codigo
        LEFT JOIN dom_qualificacoes qual ON emp.qualificacao_responsavel_cod = qual.qualificacao_codigo
        LEFT JOIN dom_cnaes        cnae ON est.cnae_fiscal_principal_cod = cnae.cnae_codigo
        LEFT JOIN dom_municipios   mun  ON est.municipio_cod            = mun.municipio_codigo
        LEFT JOIN dom_paises       pais ON est.pais_cod                 = pais.pais_codigo
        LEFT JOIN dom_motivos      mot  ON est.motivo_situacao_cadastral_cod = mot.motivo_codigo
    )
    TO '{out_file}'
    (FORMAT PARQUET)
    """

# UFs disponíveis no escopo filtrado
uf_counts = con.execute("""
    SELECT e.uf, count(*) AS n
    FROM rf_estabelecimentos e
    WHERE e.uf IS NOT NULL
      AND e.cnpj_basico IN (SELECT cnpj_basico FROM cnpjs_ativos)
    GROUP BY e.uf
    ORDER BY e.uf
""").fetchall()

print(f"  UFs no escopo filtrado: {len(uf_counts)}")
print()

t_total     = datetime.now()
grand_total = 0
errors      = []

for uf, uf_expected in uf_counts:
    out_partition = OUT_DIR / f"uf={uf}"
    out_file      = to_sql_path(out_partition / "data.parquet")

    # Idempotente por UF
    if (out_partition / "data.parquet").exists():
        n = scalar(con, f"SELECT count(*) FROM read_parquet('{out_file}')")
        grand_total += n
        print(f"  uf={uf:<3}  SKIPPED  rows={n:>8,}  cumulative={grand_total:,}")
        continue

    ensure_dir(out_partition)
    t_uf = datetime.now()

    try:
        con.execute(build_sql_uf(uf, out_file))
        n           = scalar(con, f"SELECT count(*) FROM read_parquet('{out_file}')")
        grand_total += n
        elapsed_uf  = (datetime.now() - t_uf).total_seconds()
        print(f"  uf={uf:<3}  rows={n:>8,}  time={elapsed_uf:>5.1f}s  cumulative={grand_total:,}")
    except Exception as e:
        errors.append((uf, str(e)[:100]))
        print(f"  uf={uf:<3}  ERROR: {str(e)[:100]}")

elapsed_total = (datetime.now() - t_total).total_seconds()

print()
print("=" * 60)
print(f"  Total time : {elapsed_total:.0f}s ({elapsed_total/60:.1f} min)")
print(f"  Total rows : {grand_total:,}")
print(f"  Errors     : {len(errors)}")
if errors:
    for e in errors:
        print(f"    uf={e[0]}  {e[1]}")
print("=" * 60)

In [ ]:
# =============================================================================
# Step 5 — Validation
# =============================================================================

con_val = get_connection()

suite = CheckSuite("silver_identidade")

silver_count = scalar(con_val, f"SELECT count(*) FROM read_parquet('{SD}/**/*.parquet')")

# 1. Row count matches RF establishments in scope (ADR-007 filter)
n_rf_scope = scalar(con, """
    SELECT count(*)
    FROM rf_estabelecimentos
    WHERE cnpj_basico IN (SELECT cnpj_basico FROM cnpjs_ativos)
""")
suite.add(
    "Row count matches RF scope (ADR-007)",
    silver_count == n_rf_scope,
    f"{silver_count:,} rows (RF scope: {n_rf_scope:,})"
)

# 2. cnpj_normalized integrity
nulls     = scalar(con_val, f"SELECT count(*) FROM read_parquet('{SD}/**/*.parquet') WHERE cnpj_normalized IS NULL")
suite.add("cnpj_normalized — no nulls", nulls == 0, f"{nulls} nulls")

# 3. Critical fields
for col in ["cnpj_basico", "situacao_cadastral_cod", "data_inicio_atividade"]:
    n_null = scalar(con_val, f"SELECT count(*) FROM read_parquet('{SD}/**/*.parquet') WHERE {col} IS NULL")
    suite.add(f"{col} — no nulls", n_null == 0, f"{n_null} nulls")

# 4. razao_social — LEFT JOIN miss rate (low expected)
n_null_rs = scalar(con_val, f"SELECT count(*) FROM read_parquet('{SD}/**/*.parquet') WHERE razao_social IS NULL")
pct_rs    = round(n_null_rs / silver_count * 100, 2)
suite.add("razao_social null rate < 1%", pct_rs < 1.0, f"{n_null_rs:,} nulls ({pct_rs}%)")

# 5. municipio_desc — 0 orphans expected from EDA
n_null_mun = scalar(con_val, f"SELECT count(*) FROM read_parquet('{SD}/**/*.parquet') WHERE municipio_desc IS NULL")
suite.add("municipio_desc — no nulls", n_null_mun == 0, f"{n_null_mun} nulls")

# 6. Active cnpj_basico coverage
# 33 roots (~0.008%) have no RF establishment — malformed CNPJs (e.g. '54670S76')
# or sentinels (e.g. '99999999') in contract/sanctions sources. Expected, not a bug.
n_cnpjs_silver = scalar(con_val, f"""
    SELECT count(DISTINCT LEFT(cnpj_normalized, 8))
    FROM read_parquet('{SD}/**/*.parquet')
""")
diff     = abs(n_cnpjs_silver - n_ativos)
pct_diff = round(diff / n_ativos * 100, 3)
suite.add(
    "Active cnpj_basico coverage > 99.99%",
    pct_diff < 0.01,
    f"{n_cnpjs_silver:,} in Silver vs {n_ativos:,} in scope ({diff} missing = {pct_diff}%)"
)

suite.report()
suite.assert_all_pass()

# Sample rows
print("\n  Sample rows (uf=SP, Ativas):")
sample = con_val.execute(f"""
    SELECT cnpj_normalized, razao_social, situacao_cadastral_desc,
           porte_empresa, cnae_fiscal_principal_desc,
           municipio_desc, opcao_simples, opcao_mei
    FROM read_parquet('{SD}/**/*.parquet')
    WHERE uf = 'SP' AND situacao_cadastral_cod = '02'
    LIMIT 5
""").fetchall()
for row in sample:
    def fmt(v): return (str(v) if v is not None else "NULL")[:18]
    print("    " + "  ".join(fmt(v) for v in row))

con_val.close()
con.close()

total_elapsed = (datetime.now(timezone.utc) - STARTED_AT).total_seconds()
print(f"\nNotebook completed in {total_elapsed:.0f}s ({total_elapsed/60:.1f} min)")